# 06 — Classificador v3: tamanho do texto e alinhamento de linha de base

Este notebook verifica se o tamanho do texto distorce as features de z-score máximo do notebook 05, acrescenta uma feature de alinhamento de linha de base e escolhe a configuração por validação cruzada aninhada no `train` + `val`.

**Regras definidas antes dos resultados.**
- Nenhuma feature usa as anotações. Nenhuma escolha usa o `test`.
- **Tamanho do texto:** a diferença é considerada grande se as medianas do número de componentes de texto das duas classes diferirem em 20% ou mais. Nesse caso, a versão normalizada (média dos 3 maiores |z|) entra como conjunto (c).
- **Critério de parada:** a melhor AUC externa entre todas as combinações (conjunto, modelo) precisa superar a melhor AUC externa do conjunto (a) em pelo menos 0,03. Se não superar, o `test` não é aberto, e o resultado do notebook 05 (AUC 0,626) permanece como estimativa.
- **Test:** só é aberto uma vez, depois de a configuração ser gravada em `results/configuracao_congelada.json`. O `test` já foi usado no notebook 05; uma nova avaliação nele é a segunda, e o resultado é otimista.

In [1]:
import sys
sys.path.append("../../../src")   # src/ na raiz do projeto
sys.path.append("código_v3")   # features_v3.py, arquivado nesta pasta

import json
import warnings
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

from classification import bootstrap_auc, nested_cv
from dataset_loader import load_dataset
from evaluation import wilson_score_interval
from features import FEATURE_NAMES, load_or_extract
from features_v2 import V2_NAMES, load_or_extract_v2
from features_v3 import ALIGN_NAMES, TOP3_NAMES, load_or_extract_v3

# NumPy 2.0 com a BLAS Accelerate (macOS) emite avisos espúrios de divisão por zero em
# multiplicações de matrizes; verificado no notebook 04 (diferença de 1e-16 contra um
# cálculo sem BLAS).
warnings.filterwarnings("ignore", message=".*encountered in matmul", category=RuntimeWarning)

SEED = 42
DATASET_ROOT = "../../data/findit2"
LIMIAR_TAMANHO = 0.20
GANHO_MINIMO = 0.03

## 1. Verificação do tamanho do texto

As features de z-score máximo do notebook 05 tomam o maior valor entre os componentes de uma página. O máximo de muitos valores tende a crescer com o número de componentes, então páginas com mais texto podem ter valores maiores mesmo sem adulteração. A célula abaixo conta os componentes válidos por imagem, compara as classes e mede a correlação de Spearman entre o número de componentes e cada feature de máximo.

In [2]:
amostras = {split: load_dataset(DATASET_ROOT, split) for split in ("train", "val")}
partes = []
for split, S in amostras.items():
    t = load_or_extract(S, f"../../data/features_{split}.csv")
    t = t.merge(load_or_extract_v2(S, f"../../data/features_v2_{split}.csv"), on="image", validate="one_to_one")
    t = t.merge(load_or_extract_v3(S, f"../../data/arquivo/features_v3_{split}.csv"), on="image", validate="one_to_one")
    partes.append(t)
dados = pd.concat(partes, ignore_index=True)
y = dados["is_tampered"].to_numpy()

med = dados.groupby("is_tampered")["n_components"].median()
diferenca = abs(med[1] - med[0]) / med[0]
print(f"componentes por imagem (mediana): autênticas = {med[0]:.0f} | adulteradas = {med[1]:.0f} | diferença relativa = {diferenca:.1%}")
print(dados.groupby("is_tampered")["n_components"].describe()[["mean", "25%", "50%", "75%"]].rename(index={0: "autêntica", 1: "adulterada"}).round(0))

maximos = ["ela_comp_max_z", "noise_comp_max_z", "edge_comp_max_z", "intensity_comp_max_z"]
print("\nSpearman entre n_components e cada feature:")
for maxima, top3 in zip(maximos, TOP3_NAMES):
    print(f"  {maxima:22s} rho = {spearmanr(dados['n_components'], dados[maxima])[0]:+.3f} | "
          f"{top3:22s} rho = {spearmanr(dados['n_components'], dados[top3])[0]:+.3f}")

NORMALIZA = diferenca >= LIMIAR_TAMANHO
print(f"\ndiferença entre classes >= {LIMIAR_TAMANHO:.0%}? {'sim: o conjunto (c) com a versão normalizada entra na comparação' if NORMALIZA else 'não: o conjunto (c) não é necessário'}")

aviso: 1 imagens com anotação ausente/malformada tratadas como regions=[] (train)
aviso: 2 imagens com forged divergente da anotação (train): ['X51008114260.png', 'X51006619709.png']
aviso: 2 regiões marcadas como área original mas com modificação real (train)
aviso: 1 imagens listadas em val.txt e ausentes de val/ foram ignoradas: ['X51006619709.png']
aviso: 1 regiões marcadas como área original mas com modificação real (val)


componentes por imagem (mediana): autênticas = 682 | adulteradas = 722 | diferença relativa = 5.9%
              mean    25%    50%    75%
is_tampered                            
autêntica    740.0  545.0  682.0  894.0
adulterada   752.0  534.0  722.0  878.0

Spearman entre n_components e cada feature:
  ela_comp_max_z         rho = +0.110 | ela_comp_top3_z        rho = +0.169
  noise_comp_max_z       rho = +0.068 | noise_comp_top3_z      rho = +0.101
  edge_comp_max_z        rho = -0.276 | edge_comp_top3_z       rho = -0.238
  intensity_comp_max_z   rho = +0.067 | intensity_comp_top3_z  rho = +0.082

diferença entre classes >= 20%? não: o conjunto (c) não é necessário


**Interpretação.** As medianas do número de componentes são 682 nas autênticas e 722 nas adulteradas, uma diferença de 5,9%, abaixo do limite de 20%. As correlações de Spearman com as features de máximo são fracas, de −0,276 (`edge_comp_max_z`) a +0,110 (`ela_comp_max_z`), e a versão com a média dos 3 maiores |z| tem correlações da mesma ordem. O tamanho do texto não explica a diferença entre as classes, e o conjunto (c) não foi incluído.

## 2. Alinhamento de linha de base

Um texto inserido ou um caractere colado pode ficar alguns pixels acima ou abaixo da linha em que está. A imagem é reescalada para 1000 px de largura, para que um pixel tenha o mesmo significado em todas as imagens. As linhas de texto são as faixas contínuas do perfil de projeção horizontal da máscara de Otsu (histograma das linhas da imagem, slides 232-236). Em cada linha, uma reta é ajustada às bases dos componentes, com uma rodada de remoção de discrepantes, o que tolera inclinação da digitalização e letras descendentes. O desvio de cada componente é a distância da sua base à reta, dividida pela altura mediana dos caracteres da linha.

- `align_max_z`: maior |z| robusto do desvio normalizado na página;
- `align_frac_out`: fração de componentes com |desvio| maior que 1 pixel, na escala de 1000 px.

Ressalva: se o editor alinhar bem o texto inserido, o desvio fica igual ao do texto original, e o sinal some.

In [3]:
print(dados.groupby("is_tampered")[ALIGN_NAMES].mean().rename(index={0: "autêntica", 1: "adulterada"}).T.round(3))
print("\nSpearman com n_components:")
for nome in ALIGN_NAMES:
    print(f"  {nome:16s} rho = {spearmanr(dados['n_components'], dados[nome])[0]:+.3f}")

is_tampered     autêntica  adulterada
align_max_z        76.263      65.824
align_frac_out      0.451       0.447

Spearman com n_components:
  align_max_z      rho = +0.011
  align_frac_out   rho = +0.145


**Interpretação.** As features de alinhamento não separam as classes: `align_frac_out` é 0,451 nas autênticas e 0,447 nas adulteradas, e `align_max_z` é até maior nas autênticas (76,3 contra 65,8). Os valores de `align_max_z` são muito altos porque a maior parte dos componentes fica quase exatamente sobre a linha de base, o que deixa a escala robusta pequena; caracteres de formato legítimo, como letras descendentes e sinais de pontuação, provavelmente produzem os maiores desvios. Essa explicação não foi verificada componente a componente.

## 3. Validação cruzada aninhada

Conjuntos comparados: (a) as 12 features do notebook 05; (b) as 12 mais as duas de alinhamento; (c) as 12 com as features de máximo trocadas pela média dos 3 maiores |z|, se o passo 1 indicar necessidade. Modelos: regressão logística (`class_weight="balanced"`, `StandardScaler` no pipeline) e árvore de profundidade 3.

A validação externa tem 5 folds repetidos 3 vezes (15 folds, seed 42). Em cada fold externo, toda combinação é treinada nos folds de treino e medida no fold externo (AUC externa por combinação). Além disso, uma validação interna de 5 folds, feita só com os folds de treino, escolhe uma combinação, e registra-se a AUC externa dessa escolha. Essa última estima o desempenho do processo de escolha como um todo.

In [4]:
normalizadas = [f for f in FEATURE_NAMES + V2_NAMES if f not in maximos] + TOP3_NAMES
conjuntos = {"(a) 12 atuais": FEATURE_NAMES + V2_NAMES,
             "(b) 12 + alinhamento": FEATURE_NAMES + V2_NAMES + ALIGN_NAMES}
if NORMALIZA:
    conjuntos["(c) 12 normalizadas"] = normalizadas
modelos = {
    "logística": lambda: make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=1000, random_state=SEED)),
    "árvore (prof. 3)": lambda: make_pipeline(StandardScaler(), DecisionTreeClassifier(max_depth=3, random_state=SEED)),
}
configs = {f"{c} | {m}": (dados[cols].to_numpy(), f) for c, cols in conjuntos.items() for m, f in modelos.items()}

aninhada = nested_cv(configs, y, outer_splits=5, outer_repeats=3, inner_splits=5, seed=SEED)
tabela = pd.DataFrame(aninhada["per_config"]).T
print(tabela.rename(columns={"auc_mean": "AUC externa", "auc_std": "desvio", "threshold": "limiar médio", "n_features": "n_features"}).round(3).to_string())
print(f"\nAUC externa da escolha interna: {aninhada['chosen_auc_mean']:.3f} ± {aninhada['chosen_auc_std']:.3f}")
print("combinações escolhidas pela validação interna (em 15 folds):", aninhada["chosen_counts"])

                                         AUC externa  desvio  limiar médio  n_features
(a) 12 atuais | logística                      0.665   0.047         0.497        12.0
(a) 12 atuais | árvore (prof. 3)               0.617   0.063         0.223        12.0
(b) 12 + alinhamento | logística               0.663   0.047         0.492        14.0
(b) 12 + alinhamento | árvore (prof. 3)        0.614   0.063         0.220        14.0

AUC externa da escolha interna: 0.655 ± 0.052
combinações escolhidas pela validação interna (em 15 folds): {'(a) 12 atuais | logística': 10, '(a) 12 atuais | árvore (prof. 3)': 2, '(b) 12 + alinhamento | logística': 3}


**Interpretação.** Acrescentar o alinhamento não melhora a AUC externa: 0,663 contra 0,665 com a regressão logística, e 0,614 contra 0,617 com a árvore. A validação interna escolheu o conjunto (a) com a logística em 10 dos 15 folds externos. A AUC externa da escolha interna é 0,655 ± 0,052, um pouco abaixo da melhor combinação isolada, o que é esperado: ela inclui o custo de escolher entre as combinações com poucos dados.

## 4. Critério de parada

Aplica-se a regra definida no início, sem alteração.

In [5]:
pc = aninhada["per_config"]
ordem = sorted(pc, key=lambda n: (-pc[n]["auc_mean"], pc[n]["n_features"]))
MELHOR = ordem[0]
melhor_a = max((n for n in pc if n.startswith("(a)")), key=lambda n: pc[n]["auc_mean"])
ganho = pc[MELHOR]["auc_mean"] - pc[melhor_a]["auc_mean"]
PASSA = ganho >= GANHO_MINIMO
print(f"melhor combinação: {MELHOR} | AUC externa = {pc[MELHOR]['auc_mean']:.3f}")
print(f"melhor do conjunto (a): {melhor_a} | AUC externa = {pc[melhor_a]['auc_mean']:.3f}")
print(f"ganho = {ganho:+.3f} (mínimo exigido: {GANHO_MINIMO:+.3f}) -> "
      f"{'critério atingido: congelar e avaliar no test' if PASSA else 'critério NÃO atingido: o test não é aberto'}")

melhor combinação: (a) 12 atuais | logística | AUC externa = 0.665
melhor do conjunto (a): (a) 12 atuais | logística | AUC externa = 0.665
ganho = +0.000 (mínimo exigido: +0.030) -> critério NÃO atingido: o test não é aberto


## 5. Congelamento e avaliação final

Só é executada se o critério de parada for atingido. A configuração é gravada em `results/configuracao_congelada.json` antes de qualquer leitura do `test`. O modelo é treinado no `train` + `val`, e o limiar é a média dos limiares de Youden dos folds de treino da validação externa. O IC da AUC vem de 2000 reamostragens bootstrap das imagens do `test`.

In [6]:
if PASSA:
    conj, nome_modelo = MELHOR.split(" | ")
    config = {"conjunto": conj, "features": conjuntos[conj], "modelo": nome_modelo,
              "limiar": pc[MELHOR]["threshold"], "auc_externa_cv": pc[MELHOR]["auc_mean"],
              "congelado_em": datetime.now().isoformat(timespec="seconds"), "seed": SEED}
    Path("../../results").mkdir(exist_ok=True)
    Path("../../results/configuracao_congelada.json").write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding="utf-8")
    print("configuração gravada:", json.dumps(config, ensure_ascii=False))

    # Só a partir daqui o test é lido.
    test_samples = load_dataset(DATASET_ROOT, "test")
    T = load_or_extract(test_samples, "../../data/features_test.csv")
    T = T.merge(load_or_extract_v2(test_samples, "../../data/features_v2_test.csv"), on="image", validate="one_to_one")
    T = T.merge(load_or_extract_v3(test_samples, "../../data/arquivo/features_v3_test.csv"), on="image", validate="one_to_one")
    cols = config["features"]
    modelo = modelos[nome_modelo]().fit(dados[cols].to_numpy(), y)
    prob = modelo.predict_proba(T[cols].to_numpy())[:, 1]
    yt = T["is_tampered"].to_numpy()
    pred = (prob >= config["limiar"]).astype(int)
    vn, fp, fn, vp = confusion_matrix(yt, pred, labels=[0, 1]).ravel()
    print(pd.DataFrame([[vn, fp], [fn, vp]], index=["autêntica", "adulterada"], columns=["prev. autêntica", "prev. adulterada"]))
    for nome_m, k, n in (("precisão", vp, vp + fp), ("sensibilidade", vp, vp + fn), ("especificidade", vn, vn + fp)):
        lo, hi = wilson_score_interval(k / n, n)
        print(f"{nome_m}: {k / n:.1%} (IC 95%: {lo:.1%} a {hi:.1%})")
    lo, hi = bootstrap_auc(yt, prob, n_boot=2000, seed=SEED)
    print(f"AUC: {roc_auc_score(yt, prob):.3f} (IC 95% bootstrap: {lo:.3f} a {hi:.3f})")
    print(f"baseline (tudo autêntica): acurácia = {(yt == 0).mean():.1%}")
    print("Aviso: o test foi usado nesta avaliação pela segunda vez; o resultado é otimista.")
else:
    print("Critério de parada não atingido: nenhuma configuração foi congelada e o test não foi aberto.")
    print("A estimativa vigente continua sendo a do notebook 05 (AUC 0,626 no test).")

Critério de parada não atingido: nenhuma configuração foi congelada e o test não foi aberto.
A estimativa vigente continua sendo a do notebook 05 (AUC 0,626 no test).


## 6. Conclusão

**Decisão.** O alinhamento de linha de base não melhora o classificador. A melhor AUC externa foi a do conjunto (a) com a regressão logística (0,665), e o ganho do conjunto com alinhamento foi nulo (0,663, ganho de −0,002), abaixo do mínimo de 0,03 definido antes dos resultados. Por isso, nenhuma configuração foi congelada, o arquivo `results/configuracao_congelada.json` não foi criado, e o `test` não foi aberto. A estimativa vigente continua sendo a do notebook 05: AUC de 0,626 no `test`, na única vez em que ele foi usado para esse classificador.

**Tamanho do texto.** As classes têm número de componentes parecido (diferença de 5,9% nas medianas), e as correlações com as features de máximo são fracas. O z-score máximo do notebook 05 não precisou de normalização.

**Por que o alinhamento não ajudou.** Nos dados, as features de alinhamento têm médias quase iguais nas duas classes. Duas causas prováveis, não verificadas uma a uma: (1) as ferramentas de edição usadas no dataset alinham o texto inserido à linha original, e o sinal some, como prevê a ressalva da feature; (2) o desvio máximo da página é dominado por caracteres de formato legítimo (letras descendentes, pontuação), e não por edições.

**Estimativa do processo.** A validação aninhada estima a AUC do processo completo de escolha em 0,655 ± 0,052, coerente com o 0,626 observado no `test` no notebook 05.
